In [0]:
%run ./01-config

In [0]:
import json
from confluent_kafka import Producer as KafkaProducer


class Producer():
    def __init__(self):
        Conf = Config()
        self.test_data_dir = Conf.test_data_dir
        self.landing_zone = Conf.landing_zone

        self.kafka_producer = KafkaProducer({
            "bootstrap.servers": Conf.BOOTSTRAP_SERVER,
            "security.protocol": "SASL_SSL",
            "sasl.mechanisms": "PLAIN",
            "sasl.username": Conf.CLUSTER_API_KEY,
            "sasl.password": Conf.CLUSTER_API_SECRET
        })

    def user_registration(self, set_num):
        source = (f"{self.test_data_dir}/"f"1-registered_users_{set_num}.csv")
        target = (f"{self.landing_zone}/registered_users_bz/"f"1-registered_users_{set_num}.csv")
        dbutils.fs.cp(source, target)
        print(f"Registered users copied: {set_num}")


    def gym_logins(self, set_num):
        source = (f"{self.test_data_dir}/"f"5-gym_logins_{set_num}.csv")
        target = (f"{self.landing_zone}/gym_logins_bz/"f"5-gym_logins_{set_num}.csv")
        dbutils.fs.cp(source, target)
        print(f"Gym logins copied: {set_num}")


    def publish_to_kafka(self, file_name, topic):
        file_path = f"{self.test_data_dir}/{file_name}"

        count = 0
        with open(file_path, "r") as file:
            for line in file:
                if not line.strip():
                    continue
                record = json.loads(line)
                key = str(record["key"])
                payload = record["value"]

                if isinstance(payload, str):
                    value = payload
                else:
                    value = json.dumps(payload)

                event_timestamp = int(
                    float(record["timestamp"]) * 1000
                )

                self.kafka_producer.produce(
                                                topic=topic,
                                                key=key,
                                                value=value,
                                                timestamp=event_timestamp
                                            )
                count += 1

                if count % 1000 == 0:
                    self.kafka_producer.poll(0)

        remaining = self.kafka_producer.flush(120)

        if remaining > 0:
            raise RuntimeError(f"{remaining} Kafka messages were not delivered")
        print(f"Published {count:,} records to {topic}")


    def produce(self, set_num):
        print(f"\nProducing dataset {set_num}...")
        if set_num <= 2:
            self.user_registration(set_num)
            self.gym_logins(set_num)
            self.publish_to_kafka(f"2-user_info_{set_num}.json","user_info")
            self.publish_to_kafka(f"4-workout_{set_num}.json","workout")

        if set_num <= 2:
            self.publish_to_kafka(f"3-bpm_{set_num}.json","bpm")

        print(f"Dataset {set_num} production completed.")
